In [48]:
#imports
from pyspark.sql.functions import *
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql.types import LongType, StringType, TimestampType, BooleanType
import ConnectionConfig as cc

In [72]:
#config
cc.setupEnvironment()
print(cc.config.sections())

Environment variables are set...
['default', 'tutorial_op', 'catchem', 'kafka']


In [50]:
#Cluster aanmaken
spark = cc.startLocalCluster("DIM_USER",4)
spark.getActiveSession()

In [73]:
#make connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")

In [80]:
#get info
user_src = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver" , cc.get_Property("driver")) \
    .option(
        "dbtable",
        "(select id, first_name, last_name, mail as email, street || ' ' || number as address from user_table) as subq"
    ) \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()\

user_src.write.format("delta").mode("overwrite").save("delta/USER_DIM")

In [47]:
spark.stop()